In [8]:
import requests
import pandas as pd

# ============================================================
# TESTE — API DO BANCO CENTRAL DO BRASIL (BCB/SGS)
# Série 433  → IPCA acumulado anual
# Série 4390 → Selic média anual
# ============================================================

def buscar_serie_bcb(codigo_serie, data_inicio, data_fim):
    """
    Busca uma série temporal do SGS (Sistema Gerenciador de Séries)
    do Banco Central do Brasil.
    Retorna um DataFrame com data e valor.
    """
    url = (
        f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo_serie}/dados"
        f"?formato=json&dataInicial={data_inicio}&dataFinal={data_fim}"
    )
    r = requests.get(url, timeout=30)
    df = pd.DataFrame(r.json())
    df['data']  = pd.to_datetime(df['data'], dayfirst=True)
    df['valor'] = df['valor'].str.replace(',', '.').astype(float)
    return df

# Buscar IPCA mensal (série 433) — 2014 a 2023
ipca = buscar_serie_bcb(433, '01/01/2014', '31/12/2023')
print("=== IPCA MENSAL ===")
print(ipca.head(5).to_string(index=False))
print(f"Total de meses: {len(ipca)}")

# Buscar Selic diária (série 4390) — 2014 a 2023
selic = buscar_serie_bcb(4390, '01/01/2014', '31/12/2023')
print("\n=== SELIC DIÁRIA ===")
print(selic.head(5).to_string(index=False))
print(f"Total de dias: {len(selic)}")

=== IPCA MENSAL ===
      data  valor
2014-01-01   0.55
2014-02-01   0.69
2014-03-01   0.92
2014-04-01   0.67
2014-05-01   0.46
Total de meses: 120

=== SELIC DIÁRIA ===
      data  valor
2014-01-01   0.85
2014-02-01   0.79
2014-03-01   0.77
2014-04-01   0.82
2014-05-01   0.87
Total de dias: 120


In [9]:
# ============================================================
# AGREGAR PARA ANUAL
# IPCA: acumular os meses (produto dos fatores)
# Selic: média anual das taxas mensais
# ============================================================

# IPCA acumulado anual
# Fórmula: produto de (1 + taxa_mensal/100) - 1
ipca['ano'] = ipca['data'].dt.year
ipca_anual = (
    ipca.groupby('ano')['valor']
    .apply(lambda x: ((1 + x/100).prod() - 1) * 100)
    .reset_index()
    .rename(columns={'valor': 'ipca_anual'})
)

# Selic média anual
selic['ano'] = selic['data'].dt.year
selic_anual = (
    selic.groupby('ano')['valor']
    .mean()
    .reset_index()
    .rename(columns={'valor': 'selic_media'})
)

# Juntar num único dataframe
macro = ipca_anual.merge(selic_anual, on='ano')
macro['ipca_anual']  = macro['ipca_anual'].round(2)
macro['selic_media'] = macro['selic_media'].round(2)

print("=== DADOS MACROECONÔMICOS ANUAIS 2014-2023 ===")
print(macro.to_string(index=False))


=== DADOS MACROECONÔMICOS ANUAIS 2014-2023 ===
 ano  ipca_anual  selic_media
2014        6.41         0.87
2015       10.67         1.04
2016        6.29         1.10
2017        2.95         0.79
2018        3.75         0.52
2019        4.31         0.48
2020        4.52         0.23
2021       10.06         0.36
2022        5.78         0.98
2023        4.62         1.03


In [10]:
# ============================================================
# CALCULAR SELIC ANUALIZADA E SELIC REAL
# Selic anualizada: (1 + selic_mensal/100)^12 - 1
# Selic real: desconta o IPCA da Selic nominal
# Fórmula de Fisher: (1 + selic) / (1 + ipca) - 1
# ============================================================

# Selic anualizada — converter taxa mensal para anual
macro['selic_anual'] = ((1 + macro['selic_media']/100)**12 - 1) * 100

# Selic real — retorno acima da inflação
# Isso é o custo de oportunidade real do capital
macro['selic_real'] = (
    ((1 + macro['selic_anual']/100) / (1 + macro['ipca_anual']/100)) - 1
) * 100

# Arredondar
macro['selic_anual'] = macro['selic_anual'].round(2)
macro['selic_real']  = macro['selic_real'].round(2)

print("=== MACRO COMPLETO ===")
print(macro[['ano', 'ipca_anual', 'selic_media', 'selic_anual', 'selic_real']].to_string(index=False))

=== MACRO COMPLETO ===
 ano  ipca_anual  selic_media  selic_anual  selic_real
2014        6.41         0.87        10.95        4.27
2015       10.67         1.04        13.22        2.30
2016        6.29         1.10        14.03        7.28
2017        2.95         0.79         9.90        6.75
2018        3.75         0.52         6.42        2.58
2019        4.31         0.48         5.91        1.54
2020        4.52         0.23         2.80       -1.65
2021       10.06         0.36         4.41       -5.14
2022        5.78         0.98        12.42        6.27
2023        4.62         1.03        13.08        8.09


In [ ]:
# ============================================================
# SALVAR DADOS MACROECONÔMICOS COMPLETOS
# Incluindo Selic anualizada e Selic real
# ============================================================

import os
os.makedirs("../Dados_macroeconomicos", exist_ok=True)

# Salvar arquivo completo com todas as colunas
macro.to_csv(
    "../Dados_macroeconomicos/macro_anual.csv",
    index=False,
    sep=";",
    encoding="utf-8"
)

print("=== ARQUIVO SALVO ===")
print(macro[['ano', 'ipca_anual', 'selic_media', 'selic_anual', 'selic_real']].to_string(index=False))

=== ARQUIVO SALVO ===
 ano  ipca_anual  selic_media  selic_anual  selic_real
2014        6.41         0.87        10.95        4.27
2015       10.67         1.04        13.22        2.30
2016        6.29         1.10        14.03        7.28
2017        2.95         0.79         9.90        6.75
2018        3.75         0.52         6.42        2.58
2019        4.31         0.48         5.91        1.54
2020        4.52         0.23         2.80       -1.65
2021       10.06         0.36         4.41       -5.14
2022        5.78         0.98        12.42        6.27
2023        4.62         1.03        13.08        8.09
